# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Tim Schmale | |
| Member|Lennart Oberkönig | |
| Member *(optional)* | | |


## 2. Mission Title & Research Question

**Title:** *What Drives Negative Price Events in the DE-LU Day-Ahead Electricity Market?*

**Research question:**  
Under which market conditions do negative day-ahead electricity prices occur in the DE-LU bidding zone, and how strongly are these events associated with external factors such as weather and seasonality?

**Why it matters:**  

European electricity markets have changed substantially due to the increasing integration of renewable energy sources. Especially in Germany, wind and solar generation play a central role in wholesale price formation, as they have very low marginal costs and are prioritized in market dispatch. This can be described as merit-order effect, whitch is based on the idea that the demand for electricity is quasi inelastic and that renewable feed-in can therefore reduce prices by replacing conventional generation. (Gürtler und Paulsen, 2018, P. 150)

In this project, we focus on negative day-ahead prices in the German-Luxembourg electricity bidding zone. Negative prices are economically relevant because they indicate periods of excess supply, limited flexibility and changing market incentives (Seel et al., 2021, P. 3). Understanding the market conditions under which these events occur is important for supply and demand management. Thus, we investigate how negative price events are associated with renewable generation, weather-driven supply conditions and seasonality.

In order to forecast, if the day-ahead price of the next hour is negative, we begin with data investigation and preparation, enrichting the power system data with weather information. After that we investigate the underlying causal stuctures of the data. Next, we apply k-means clustering as an unsupervised learning approach to identify different market situations. Based on these insides we build a random forest classifier as our baseline model. Finally we challenge these results with a neural network as a state of the art modelling approach.

## 3. Data

**Source(s):**  
Open Power System Data
Open Power System Data is provided by Neon Neue Energieökonomik, Technical University of Berlin, ETH Zürich and DIW Berlin - Wiese et al. 2019 
https://data.open-power-system-data.org/time_series/
Weather Data
Weather is provided by Neon Neue Energieökonomik, Technical University of Berlin, ETH Zürich and DIW Berlin - Wiese et al. 2019 
https://data.open-power-system-data.org/weather_data/


**Unit of observation:** One row represents one hour in the German-Luxembourg electricity bidding zone.

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
|cet_cest_timestamp|datetime|Identifier | Start of timeperiod in Central European (Summer-) Time|
|DE_LU_price_day_ahead|numeric| target | Day-ahead spot price for DE-LU (bidding zone) in EUR|
|DE_LU_solar_generation_actual|numeric| feature | Actual solar generation in DE-LU (bidding zone) in MW|
|DE_LU_wind_generation_actual|numeric| feature | Actual wind generation in DE-LU (bidding zone) in MW|
|DE_LU_load_actual_entsoe_transparency|numeric| feature | Total load in DE-LU (bidding zone) in MW as published on ENTSO-E Transparency Platform|
|hour|integer| feature | Hour of the day|
|weekday|string| feature | Day of the week|
|month|string| feature | Month of the year|
|DE_temperature|float| feature | Temperature weather variable for DE in degrees C|
|DE_radiation_direct_horizontal|float| feature | Radiation_direct_horizontal weather variable for DE in W/m2|
|DE_radiation_diffuse_horizontal|float| feature | Radiation_diffuse_horizontal weather variable for DE in W/m2|





**Potential data quality issues:**  
1. Missing data for renewable energy capacity of Luxembourg
2. Missing Values
3. Unequal distribution of positive and negative prices


**Reading, joining and transforming data**

In [96]:
import pandas as pd
import matplotlib.pyplot as plt

opsd = "/Users/lennart/coding/DAI/time_series_60min_singleindex.csv"
weather = "/Users/lennart/coding/DAI/weather_data.csv"

opsd = pd.read_csv(opsd,delimiter=";")
weather = pd.read_csv(weather,delimiter=",")

opsd = opsd[[
    "utc_timestamp",
    "cet_cest_timestamp",
    "DE_LU_price_day_ahead",
    "DE_LU_solar_generation_actual",
    "DE_LU_wind_generation_actual",
    "DE_LU_load_actual_entsoe_transparency",
    "DE_wind_capacity",
    "DE_solar_capacity"]]



weather = weather[[
    "utc_timestamp",
    "DE_temperature",
    "DE_radiation_direct_horizontal",
    "DE_radiation_diffuse_horizontal"]]


df = pd.merge(opsd, weather, on="utc_timestamp", how="inner")


df = df[df["cet_cest_timestamp"]>="2018-09-30T23:00:00Z"]
df["cet_cest_timestamp"] = pd.to_datetime(df["cet_cest_timestamp"], utc=True).dt.tz_convert("Europe/Berlin")
cols = [
    "DE_LU_solar_generation_actual",
    "DE_LU_wind_generation_actual",
    "DE_LU_load_actual_entsoe_transparency",
    "DE_wind_capacity",
    "DE_solar_capacity"]


for col in cols:
    df[col] = df[col].str.replace(".", "").astype(float)

df["DE_LU_price_day_ahead"] = pd.to_numeric(df["DE_LU_price_day_ahead"], errors='coerce')
df

/var/folders/j7/61_hz7955mn3z2j247bwq_m00000gn/T/ipykernel_88250/734959595.py:7: DtypeWarning: Columns (0: AT_price_day_ahead, 1: BG_load_forecast_entsoe_transparency, 2: CH_solar_capacity, 3: CH_solar_generation_actual, 4: CY_load_actual_entsoe_transparency, 5: CY_load_forecast_entsoe_transparency, 6: CY_wind_onshore_generation_actual, 7: DE_solar_capacity, 8: DE_wind_capacity, 9: DE_wind_offshore_capacity, 10: DE_wind_onshore_capacity, 11: DE_50hertz_wind_offshore_generation_actual, 12: DE_LU_load_actual_entsoe_transparency, 13: DE_LU_load_forecast_entsoe_transparency, 14: DE_LU_price_day_ahead, 15: DE_LU_solar_generation_actual, 16: DE_LU_wind_generation_actual, 17: DE_LU_wind_offshore_generation_actual, 18: DE_LU_wind_onshore_generation_actual, 19: DK_solar_capacity, 20: DK_wind_capacity, 21: DK_wind_offshore_capacity, 22: DK_wind_onshore_capacity, 23: DK_1_price_day_ahead, 24: DK_2_price_day_ahead, 25: DK_2_solar_generation_actual, 26: GB_GBN_price_day_ahead, 27: GB_GBN_solar_capa

,utc_timestamp,cet_cest_timestamp,DE_LU_price_day_ahead,DE_LU_solar_generation_actual,DE_LU_wind_generation_actual,DE_LU_load_actual_entsoe_transparency,DE_wind_capacity,DE_solar_capacity,DE_temperature,DE_radiation_direct_horizontal,DE_radiation_diffuse_horizontal
32855,2018-09-30T22:00:00Z,2018-10-01 00:00:00+02:00,NaN,NaN,59320000.0,NaN,477300000.0,460990000.0,8.676,0.0,0.0
32856,2018-09-30T23:00:00Z,2018-10-01 01:00:00+02:00,561.0,NaN,60420000.0,NaN,477300000.0,460990000.0,8.258,0.0,0.0
32857,2018-10-01T00:00:00Z,2018-10-01 02:00:00+02:00,514.1,NaN,60210000.0,418740000.0,477300000.0,460990000.0,7.889,0.0,0.0
32858,2018-10-01T01:00:00Z,2018-10-01 03:00:00+02:00,473.8,NaN,63420000.0,427130000.0,477300000.0,460990000.0,7.620,0.0,0.0
32859,2018-10-01T02:00:00Z,2018-10-01 04:00:00+02:00,475.9,NaN,71440000.0,441650000.0,477300000.0,460990000.0,7.395,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
43820,2019-12-31T19:00:00Z,2019-12-31 20:00:00+01:00,422.0,0.0,88750000.0,479280000.0,NaN,NaN,0.767,0.0,0.0
43821,2019-12-31T20:00:00Z,2019-12-31 21:00:00+01:00,397.4,0.0,76520000.0,462350000.0,NaN,NaN,0.656,0.0,0.0
43822,2019-12-31T21:00:00Z,2019-12-31 22:00:00+01:00,388.8,0.0,72830000.0,458710000.0,NaN,NaN,0.476,0.0,0.0
43823,2019-12-31T22:00:00Z,2019-12-31 23:00:00+01:00,373.9,0.0,65730000.0,443680000.0,NaN,NaN,0.226,0.0,0.0


**1. Missing data for renewable energy capacity of Luxembourg**

The data of Luxembourgs renewable energy capacity (solar and wind) is not available. To be able to capture the growth of the renewable energy capacity, we use the German capacity as a proxy for the German-Luxembourg electricity bidding zone.

**2. Missing Values**

In [106]:
df.isna().sum()

utc_timestamp                             0
cet_cest_timestamp                        0
DE_LU_price_day_ahead                    23
DE_LU_solar_generation_actual             7
DE_LU_wind_generation_actual              0
DE_LU_load_actual_entsoe_transparency    22
DE_wind_capacity                          0
DE_solar_capacity                         0
DE_temperature                            0
DE_radiation_direct_horizontal            0
DE_radiation_diffuse_horizontal           0
dtype: int64

To address the missing values in the different columns we use different appoaches based on the context.


| Column | Approach | 
|----------|------|
|DE_LU_price_day_ahead| k-Nearest Neighbors regression|
|DE_LU_solar_generation_actual| k-Nearest Neighbors regression|
|DE_LU_load_actual_entsoe_transparency| k-Nearest Neighbors regression|
|DE_wind_capacity| fill forward|
|DE_solar_capacity| fill forward|

In [98]:
df[df[df.columns].isna().any(axis=1)]

,utc_timestamp,cet_cest_timestamp,DE_LU_price_day_ahead,DE_LU_solar_generation_actual,DE_LU_wind_generation_actual,DE_LU_load_actual_entsoe_transparency,DE_wind_capacity,DE_solar_capacity,DE_temperature,DE_radiation_direct_horizontal,DE_radiation_diffuse_horizontal
32855,2018-09-30T22:00:00Z,2018-10-01 00:00:00+02:00,NaN,NaN,59320000.0,NaN,477300000.0,460990000.0,8.676,0.0,0.0
32856,2018-09-30T23:00:00Z,2018-10-01 01:00:00+02:00,561.0,NaN,60420000.0,NaN,477300000.0,460990000.0,8.258,0.0,0.0
32857,2018-10-01T00:00:00Z,2018-10-01 02:00:00+02:00,514.1,NaN,60210000.0,418740000.0,477300000.0,460990000.0,7.889,0.0,0.0
32858,2018-10-01T01:00:00Z,2018-10-01 03:00:00+02:00,473.8,NaN,63420000.0,427130000.0,477300000.0,460990000.0,7.620,0.0,0.0
32859,2018-10-01T02:00:00Z,2018-10-01 04:00:00+02:00,475.9,NaN,71440000.0,441650000.0,477300000.0,460990000.0,7.395,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
43820,2019-12-31T19:00:00Z,2019-12-31 20:00:00+01:00,422.0,0.0,88750000.0,479280000.0,NaN,NaN,0.767,0.0,0.0
43821,2019-12-31T20:00:00Z,2019-12-31 21:00:00+01:00,397.4,0.0,76520000.0,462350000.0,NaN,NaN,0.656,0.0,0.0
43822,2019-12-31T21:00:00Z,2019-12-31 22:00:00+01:00,388.8,0.0,72830000.0,458710000.0,NaN,NaN,0.476,0.0,0.0
43823,2019-12-31T22:00:00Z,2019-12-31 23:00:00+01:00,373.9,0.0,65730000.0,443680000.0,NaN,NaN,0.226,0.0,0.0


In [105]:
df.sort_values("cet_cest_timestamp", inplace=True)

df[["DE_wind_capacity","DE_solar_capacity"]]= df[["DE_wind_capacity","DE_solar_capacity"]].ffill()

In [107]:
from sklearn.neighbors import KNeighborsRegressor


impute_targets = [
    "DE_LU_load_actual_entsoe_transparency",
    "DE_LU_solar_generation_actual",
    "DE_LU_price_day_ahead",
]

predictor_cols = [
    "DE_LU_wind_generation_actual",
    "DE_wind_capacity",
    "DE_solar_capacity",
    "DE_temperature",
    "DE_radiation_direct_horizontal",
    "DE_radiation_diffuse_horizontal",
]


for target in impute_targets:
    valid_predictors = df[predictor_cols].notna().all(axis=1)
    train_mask = df[target].notna() & valid_predictors
    predict_mask = df[target].isna() & valid_predictors

    X_train = df.loc[train_mask, predictor_cols]
    y_train = df.loc[train_mask, target]
    X_pred = df.loc[predict_mask, predictor_cols]

    if len(X_train) == 0 or len(X_pred) == 0:
        continue

    knn = KNeighborsRegressor(n_neighbors=5, weights="distance")
    knn.fit(X_train, y_train)
    df.loc[predict_mask, target] = knn.predict(X_pred)


Imputed 22 missing values in DE_LU_load_actual_entsoe_transparency using 10948 training rows.
Imputed 7 missing values in DE_LU_solar_generation_actual using 10963 training rows.
Imputed 23 missing values in DE_LU_price_day_ahead using 10947 training rows.


In [108]:
df.isna().sum()

utc_timestamp                            0
cet_cest_timestamp                       0
DE_LU_price_day_ahead                    0
DE_LU_solar_generation_actual            0
DE_LU_wind_generation_actual             0
DE_LU_load_actual_entsoe_transparency    0
DE_wind_capacity                         0
DE_solar_capacity                        0
DE_temperature                           0
DE_radiation_direct_horizontal           0
DE_radiation_diffuse_horizontal          0
dtype: int64

**3. Unequal distribution of positive and negative prices**


In [116]:
df

,utc_timestamp,cet_cest_timestamp,DE_LU_price_day_ahead,DE_LU_solar_generation_actual,DE_LU_wind_generation_actual,DE_LU_load_actual_entsoe_transparency,DE_wind_capacity,DE_solar_capacity,DE_temperature,DE_radiation_direct_horizontal,DE_radiation_diffuse_horizontal,negative_price_flagg,negative_price_flag
32855,2018-09-30T22:00:00Z,2018-10-01 00:00:00+02:00,506.214782,1.575279e+08,59320000.0,5.221296e+08,477300000.0,460990000.0,8.676,0.0,0.0,0,0
32856,2018-09-30T23:00:00Z,2018-10-01 01:00:00+02:00,561.000000,1.817662e+08,60420000.0,4.955749e+08,477300000.0,460990000.0,8.258,0.0,0.0,0,0
32857,2018-10-01T00:00:00Z,2018-10-01 02:00:00+02:00,514.100000,1.775332e+08,60210000.0,4.187400e+08,477300000.0,460990000.0,7.889,0.0,0.0,0,0
32858,2018-10-01T01:00:00Z,2018-10-01 03:00:00+02:00,473.800000,1.324214e+08,63420000.0,4.271300e+08,477300000.0,460990000.0,7.620,0.0,0.0,0,0
32859,2018-10-01T02:00:00Z,2018-10-01 04:00:00+02:00,475.900000,8.140661e+07,71440000.0,4.416500e+08,477300000.0,460990000.0,7.395,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43820,2019-12-31T19:00:00Z,2019-12-31 20:00:00+01:00,422.000000,0.000000e+00,88750000.0,4.792800e+08,504520000.0,505080000.0,0.767,0.0,0.0,0,0
43821,2019-12-31T20:00:00Z,2019-12-31 21:00:00+01:00,397.400000,0.000000e+00,76520000.0,4.623500e+08,504520000.0,505080000.0,0.656,0.0,0.0,0,0
43822,2019-12-31T21:00:00Z,2019-12-31 22:00:00+01:00,388.800000,0.000000e+00,72830000.0,4.587100e+08,504520000.0,505080000.0,0.476,0.0,0.0,0,0
43823,2019-12-31T22:00:00Z,2019-12-31 23:00:00+01:00,373.900000,0.000000e+00,65730000.0,4.436800e+08,504520000.0,505080000.0,0.226,0.0,0.0,0,0


In [121]:
df["negative_price_flag"] = (df["DE_LU_price_day_ahead"] < 0).astype(int)
df.groupby("negative_price_flag").count()["cet_cest_timestamp"]

negative_price_flag
0    10732
1      238
Name: cet_cest_timestamp, dtype: int64

Take away: A high class inbalance is apparent. This needs to be handled in the modelling part by balancing out the classes by using e.g. SMOTE or using a the correct evaluation metric (sensitivity instead of accuracy). Otherwise the modell will classify all observations as positive.

## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference
- [x] Causal graph / DAG (DoWhy)
- [ ] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

*Justification:* Gaining an deep understanding of the inference and causallity of the different variables.

### 4b. Supervised Learning
- [ ] Linear / Ridge / Lasso regression
- [ ] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [x] Decision Tree / Random Forest
- [x] Neural network (regression or classification)
- [ ] Other: ___

*Justification:* Using one simpler method as a baseline and comparing against the state of the art.

### 4c. Unsupervised Learning / Generative Models
- [x] K-Means clustering
- [ ] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification:* 


## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

Unsupervised Learning
- Usnig silhouette score to evaluate the clustering process

Supervised Learning
- Comparing the modell against random classification
- Using the evaluation metric of sensivity of specificly identify negative prices
- Comparing Neural Network with Random Forrest



## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | | Data collection & cleaning |
| 2 | | Feature Construction |
| 3 | | Causal inference block |
| 4 | | Supervised learning block |
| 5 | | Unsupervised |
| 6 | | Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
